In [9]:
!git clone https://github.com/xchlai/DropCount.git

Cloning into 'DropCount'...
fatal: unable to access 'https://github.com/xchlai/DropCount.git/': Could not resolve host: github.com


In [ ]:
%cd DropCount

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Cell 3: verify device
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%%bash
cd /kaggle/working/DropCount

cat > configs/one_setting_10k.yaml <<'YAML'
simulation:
  n_droplets: 10000
  true_copy_range: [0, 20000]
  copy_sampling_mode: uniform_integer
  simulation_mode: fixed_total_multinomial
  random_seed: 123
  false_positive_rate_range: [0.0, 0.05]

  train_samples_per_epoch: 10000
  val_samples: 1000
  test_samples_per_combo: 10000
  test_n_droplets: [10000]

  copy_bins: [0, 20, 100, 1000, 5000, 10000, 20000]

  distributions:
    name: truncated_normal
    cv: 0.5
    train_names: [truncated_normal]
    eval_names: [truncated_normal]
    cv_values: [0.5]
    lognormal_sigma_cap: 2.0
    binomial_k: 10
    positive_floor: 1.0e-6

model:
  model_type: perceiver
  input_dim: 10
  hidden_dim: 96
  latent_dim: 96
  num_latents: 16
  num_heads: 4
  num_self_attn_layers: 2
  dropout: 0.1
  fourier_features: 2
  use_rmsnorm: false

training:
  batch_size: 1
  epochs: 200
  learning_rate: 3.0e-4
  weight_decay: 1.0e-4
  grad_clip_norm: 1.0
  loss_name: huber_log
  linear_loss_weight: 0.05
  huber_delta: 0.5
  early_stopping_patience: 20
  num_workers: 0
  amp: true
  device: auto

output:
  run_root: outputs
  run_name: one_setting_10k
  save_validation_dataset: true

baselines:
  max_copy_cap: 1000000.0
  eps: 1.0e-12
  mle_search_upper_multiplier: 20.0
YAML

In [ ]:
!python train.py --config configs/one_setting_10k.yaml --run-name one_setting_10k

In [ ]:
!python evaluate.py --config configs/one_setting_10k.yaml --run-dir outputs/one_setting_10k